# MultilingualLiterary on Llama-3.1-8B

Native author-balanced literary prose, same protocol as `Corpus_Expansion_Llama` (100-token context, 30-token target, 3 positions per text, intact + token-shuffled marginals). Outputs cache files **into the corpus_expansion cache dir** so `analysis/reconcile_metrics_corpus_expansion.py` picks them up alongside the en quartet without modification.

**Drive layout expected:**
- `My Drive/LRTIA/Data/multilingual_literary/manifests/<lang>.json`
- `My Drive/LRTIA/Data/multilingual_literary/data/<lang>/<author>/<text>.txt`

**Outputs:** `My Drive/LRTIA/Results/corpus_expansion/llama/literary_<lang>.json`

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, random, time
from pathlib import Path
from scipy import stats
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/LRTIA')
DATA = DRIVE / 'Data/multilingual_literary'
# Save into the corpus_expansion cache dir so the reconcile script picks the literary
# corpora up alongside the en quartet without any modification.
BASE = DRIVE / 'Results/corpus_expansion/llama'
BASE.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
C = 100              # context length
TARGET_LEN = 30
TARGET_FRACS = [0.25, 0.50, 0.75]
MIN_TOTAL_TOKENS = 200   # need enough for 100 ctx + 30 tgt + buffer for positions
N_SHUFFLES = 1
SEED = 20260501

# Pick which languages to run. ko/tr are too thin per the MultilingualLiterary README
# but included here as single/few-author baselines; exclude them for the headline claim.
RUN_LANGS = ['ja', 'fi', 'ko', 'tr']

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Languages: {RUN_LANGS}')

In [ ]:
# Load Llama once.
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map='auto'
)
model.eval()
print('Model loaded')

In [ ]:
# Pipeline functions — identical to Corpus_Expansion_Llama so cache files merge cleanly.
@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2:
        return float('inf'), float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i + 1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0:
        return float('inf'), float('inf')
    mn = nll / cnt
    return math.exp(mn), mn

def compute_corrected_curves(ctx, tgt):
    mc = len(ctx)
    o_ppl, o_nll, s_ppl, s_nll = [], [], [], []
    for c in range(mc + 1):
        pfx = ctx[-c:] if c > 0 else []
        p, n = ppl_nll(pfx, tgt)
        o_ppl.append(p); o_nll.append(n)
        if c == 0:
            s_ppl.append(p); s_nll.append(n)
        else:
            rng = random.Random(SEED + c)
            sp_l, sn_l = [], []
            for _ in range(N_SHUFFLES):
                sh = list(pfx); rng.shuffle(sh)
                sp, sn = ppl_nll(sh, tgt)
                if not math.isinf(sp):
                    sp_l.append(sp); sn_l.append(sn)
            s_ppl.append(np.mean(sp_l) if sp_l else p)
            s_nll.append(np.mean(sn_l) if sn_l else n)
    dists = list(range(1, mc + 1))
    mo = [o_ppl[d - 1] - o_ppl[d] for d in dists]
    ms = [s_ppl[d - 1] - s_ppl[d] for d in dists]
    delta = [a - b for a, b in zip(mo, ms)]
    mo_n = [o_nll[d - 1] - o_nll[d] for d in dists]
    ms_n = [s_nll[d - 1] - s_nll[d] for d in dists]
    delta_n = [a - b for a, b in zip(mo_n, ms_n)]
    return {
        'distances': dists,
        'ordered_ppl': o_ppl, 'shuffled_ppl': s_ppl, 'delta_ppl': delta,
        'ordered_nll': o_nll, 'shuffled_nll': s_nll, 'delta_nll': delta_n,
    }

BIN_EDGES = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]
def fit_pl(marg):
    bm, bc = [], []
    for i in range(len(BIN_EDGES) - 1):
        lo, hi = BIN_EDGES[i], BIN_EDGES[i + 1]
        vals = marg[lo - 1:hi - 1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals)); bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        s, _, r, _, _ = stats.linregress(np.log(bc), np.log(bm))
        return s, r
    return None, None

print('Pipeline functions ready')

In [ ]:
# Build per-text targets from the MultilingualLiterary manifests.
def iter_targets(lang):
    """Yield (corpus_id, document_id, language, family, text_path, target_frac) tuples."""
    m = json.loads((DATA / 'manifests' / f'{lang}.json').read_text(encoding='utf-8'))
    family = m.get('family', '')
    corpus_id = f'literary_{lang}'
    for author in m['authors']:
        for t in author['texts']:
            text_path = DATA / t['text_path']
            doc_id = t['text_id']
            for frac in TARGET_FRACS:
                yield {
                    'corpus_id': corpus_id,
                    'document_id': doc_id,
                    'author_id': author['author_id'],
                    'language': lang,
                    'family': family,
                    'genre': 'literary_narrative',
                    'modality': 'written',
                    'text_path': str(text_path),
                    'target_frac': frac,
                }

# Quick preview: how many texts per language pass the length floor?
for lang in RUN_LANGS:
    targets = list(iter_targets(lang))
    docs = sorted({t['document_id'] for t in targets})
    print(f'  {lang}: {len(docs)} docs × 3 targets = {len(targets)} target slots')

In [ ]:
# Main run loop — one cache JSON per language, same format as corpus_expansion.
for lang in RUN_LANGS:
    corpus_id = f'literary_{lang}'
    cache_path = BASE / f'{corpus_id}.json'
    if cache_path.exists():
        with open(cache_path) as f: n = len(json.load(f))
        print(f'\n{corpus_id}: cached ({n})'); continue

    targets = list(iter_targets(lang))
    if not targets:
        print(f'\n{corpus_id}: no targets'); continue

    print(f'\n{"="*60}\n{corpus_id} ({len(targets)} target slots)\n{"="*60}')

    by_doc = {}
    for t in targets:
        by_doc.setdefault(t['document_id'], []).append(t)

    t0 = time.time()
    results = []
    skipped = 0

    for doc_id in tqdm(by_doc, desc=corpus_id):
        doc_meta = by_doc[doc_id][0]
        text = Path(doc_meta['text_path']).read_text(encoding='utf-8', errors='replace').strip()
        full_ids = tokenizer.encode(text, add_special_tokens=False)
        n_tok = len(full_ids)
        if n_tok < MIN_TOTAL_TOKENS:
            skipped += 1; continue

        # Target windows: pick at fractions of the *available* range so 100-token context
        # fits before each target.
        rem_start = C
        rem_end = n_tok - TARGET_LEN
        if rem_end <= rem_start:
            skipped += 1; continue

        for t in by_doc[doc_id]:
            frac = t['target_frac']
            ts = int(rem_start + frac * (rem_end - rem_start))
            te = ts + TARGET_LEN
            cs = ts - C; ce = ts
            if cs < 0 or te > n_tok:
                continue
            ctx = full_ids[cs:ce]
            tgt = full_ids[ts:te]
            if len(ctx) < C or len(tgt) < 5:
                continue

            r = compute_corrected_curves(ctx, tgt)
            r['corpus_id'] = corpus_id
            r['document_id'] = doc_id
            r['author_id'] = t['author_id']
            r['target_id'] = f"{doc_id}__pos{int(frac*100):02d}"
            r['target_frac'] = frac
            r['language'] = lang
            r['family'] = doc_meta['family']
            r['genre'] = 'literary_narrative'
            r['modality'] = 'written'
            results.append(r)

    elapsed = time.time() - t0
    with open(cache_path, 'w') as f:
        json.dump(results, f)

    print(f'  {len(results)} results in {elapsed/60:.1f} min ({skipped} docs skipped)')

    if results:
        mean_delta = float(np.mean([np.mean(r['delta_ppl']) for r in results]))
        curve = np.mean([r['delta_ppl'] for r in results], axis=0)
        alpha, r_val = fit_pl(np.array(curve))
        a_str = f'{alpha:.3f} (r={r_val:.3f})' if alpha else '—'
        print(f'  Mean Delta: {mean_delta:.4f}, alpha: {a_str}')

print('\nAll done.')